# Phase 1 - Replay Original ARCO-ERA5 Records Through Kafka

This notebook reads a representative slice from the original January 2024 MinIO partition with Spark and replays those records through Kafka. The complete source partition remains immutable in MinIO.

A bounded replay is deliberate: Kafka demonstrates streaming behavior while Spark batch jobs process the large source partitions directly.

In [1]:
import json
import os
import time
import sys
from pathlib import Path
from datetime import timezone

from kafka import KafkaProducer
from pyspark.sql import functions as F

PROJECT_ROOT = Path("/workspace")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spark_jobs.common import create_spark_session, load_settings, normalized_timestamp, s3a_uri

YEAR, MONTH = 2024, 1
REPLAY_MAX_RECORDS = int(os.getenv("KAFKA_REPLAY_MAX_RECORDS", "240"))
REPLAY_DELAY_SECONDS = float(os.getenv("KAFKA_REPLAY_DELAY_SECONDS", "0.03"))
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_WEATHER_RAW", "weather.raw")
RUN_LIVE_REPLAY = False  # Set True only when demonstrating new Kafka events.

settings = load_settings()
spark = create_spark_session("phase1-original-weather-kafka-replay", settings)
source = s3a_uri(settings.raw_bucket, f"arco_era5_us_airport_hourly/year={YEAR}/month={MONTH:02d}")
print("Original source:", source)

Original source: s3a://raw/arco_era5_us_airport_hourly/year=2024/month=01


## Select a Representative Original Slice

The replay includes multiple airports and hours from the original monthly partition. Unit conversions happen before publishing so streamed events are human-readable.

In [2]:
raw_df = spark.read.parquet(source)
replay_df = (
    raw_df
    .withColumn("timestamp_utc", normalized_timestamp(raw_df, "time_utc"))
    .withColumn("temperature_c", F.col("2m_temperature") - F.lit(273.15))
    .withColumn(
        "wind_speed_kts",
        F.sqrt(F.pow(F.col("10m_u_component_of_wind"), 2) + F.pow(F.col("10m_v_component_of_wind"), 2)) * F.lit(1.94384),
    )
    .withColumn("precipitation_mm", F.col("total_precipitation") * F.lit(1000.0))
    .filter((F.col("day") == 1) & (F.col("hour_utc") == 0))
    .select(
        F.col("airport_key").cast("int"),
        F.col("timestamp_utc").cast("string"),
        F.round("temperature_c", 3).alias("temperature_c"),
        F.round("wind_speed_kts", 3).alias("wind_speed_kts"),
        F.round("precipitation_mm", 6).alias("precipitation_mm"),
        F.col("surface_pressure").cast("double").alias("surface_pressure_pa"),
    )
    .orderBy("timestamp_utc", "airport_key")
    .limit(REPLAY_MAX_RECORDS)
)
replay_df.printSchema()
replay_df.show(10, truncate=False)
print("Replay records:", replay_df.count())

root
 |-- airport_key: integer (nullable = true)
 |-- timestamp_utc: string (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- wind_speed_kts: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- surface_pressure_pa: double (nullable = true)



+-----------+-------------------+-------------+--------------+----------------+-------------------+
|airport_key|timestamp_utc      |temperature_c|wind_speed_kts|precipitation_mm|surface_pressure_pa|
+-----------+-------------------+-------------+--------------+----------------+-------------------+
|0          |2024-01-01 00:00:00|5.094        |1.773         |0.008386        |101263.828125      |
|1          |2024-01-01 00:00:00|-0.444       |3.473         |0.0             |90422.6015625      |
|2          |2024-01-01 00:00:00|-2.58        |19.867        |0.648143        |99646.59375        |
|3          |2024-01-01 00:00:00|9.621        |6.072         |0.0             |99181.625          |
|4          |2024-01-01 00:00:00|-4.281       |8.319         |0.0             |99410.71875        |
|5          |2024-01-01 00:00:00|4.754        |8.562         |0.0             |97940.2734375      |
|6          |2024-01-01 00:00:00|9.16         |3.973         |0.0             |86662.9140625      |


Replay records: 240


## Publish to Kafka

Start the consumer notebook first so the event flow is visible while this cell executes.

In [3]:
if RUN_LIVE_REPLAY:
    producer = KafkaProducer(
    bootstrap_servers=os.environ["KAFKA_BOOTSTRAP_SERVERS"],
    value_serializer=lambda event: json.dumps(event, default=str).encode("utf-8"),
    )

    sent = 0
    for row in replay_df.toLocalIterator():
        event = row.asDict(recursive=True)
        producer.send(KAFKA_TOPIC, key=str(event["airport_key"]).encode("utf-8"), value=event)
        sent += 1
        if sent % 50 == 0:
            print("Published events:", sent)
        time.sleep(REPLAY_DELAY_SECONDS)

    producer.flush()
    producer.close()
    print("Published original ARCO-ERA5 events:", sent)
else:
    print("Safe presentation mode: replay preview only.")
    print("Set RUN_LIVE_REPLAY=True after starting the consumer notebook to publish events.")
spark.stop()
print("Kafka replay notebook complete.")

Safe presentation mode: replay preview only.
Set RUN_LIVE_REPLAY=True after starting the consumer notebook to publish events.


Kafka replay notebook complete.
